In [1]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'



In [2]:
from models import Ride, ride_deserializer

In [3]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='green-trips-hw',
    value_deserializer=ride_deserializer,
    consumer_timeout_ms=10000
)

In [2]:
import psycopg2

conn = psycopg2.connect(
    host='localhost', port=5432, database='postgres', user='postgres', password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("TRUNCATE TABLE green_trips_events;")
print("Table successfully cleared!")

cur.close()
conn.close()

Table successfully cleared!


In [ ]:


print(f"Listening to {topic_name} and writing to PostgreSQL...")

long_trips_count = 0
for message in consumer:
    ride = message.value
    if ride.trip_distance > 5.0:
        long_trips_count += 1
    
    cur.execute(
        """INSERT INTO green_trips_events
           (lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, 
            passenger_count, trip_distance, tip_amount, total_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.lpep_pickup_datetime, ride.lpep_dropoff_datetime, 
         ride.PULocationID, ride.DOLocationID, ride.passenger_count, 
         ride.trip_distance, ride.tip_amount, ride.total_amount))

    if long_trips_count % 100 == 0:
        print(f"Found {long_trips_count} long trips...")

print("\n--- FINISHED ---")
print(f"Total trips with distance > 5.0: {long_trips_count}")
consumer.close()


Listening to green-trips and writing to PostgreSQL...
Found 0 long trips...
Found 0 long trips...
Found 0 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 100 long trips...
Found 200 long trips...
Found 300 long trips...
Found 400 long trips...
Found 400 long trips...
Found 400 long trips...
Found 400 long trips...
Found 400 long trips...
Found 500 long trips...
Found 500 long trips...
Found 500 long trips...
Found 500 long trips...
Found 500 long trips...
Found 500 long trips...
Found 600 long trips...
Found 600 long trips...
Found 600 long trips...
Found 700 long trips...
Found 700 long t

In [3]:
from kafka import KafkaConsumer
from models import ride_deserializer
import psycopg2
import uuid

# 1. Open the database connection
conn = psycopg2.connect(
    host='localhost', port=5432, database='postgres', user='postgres', password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

# 2. Connect to the CLEAN topic with a RANDOM group_id to reset the bookmark
topic_name = 'green-trips-fresh' # Make sure this matches your clean topic!
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    group_id=f"db-writer-{uuid.uuid4()}", # Random ID forces it to start at row 1
    value_deserializer=ride_deserializer,
    consumer_timeout_ms=10000
)

print(f"Reading from '{topic_name}' and writing to Postgres...")

long_trips_count = 0
inserted_rows = 0

# 3. Process the messages
for message in consumer:
    ride = message.value
    
    # Homework Check
    if ride.trip_distance > 5.0:
        long_trips_count += 1
        
    # Database Insert
    cur.execute(
        """INSERT INTO green_trips_events
           (lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, 
            passenger_count, trip_distance, tip_amount, total_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.lpep_pickup_datetime, ride.lpep_dropoff_datetime, 
         ride.PULocationID, ride.DOLocationID, ride.passenger_count, 
         ride.trip_distance, ride.tip_amount, ride.total_amount)
    )
    
    inserted_rows += 1

print("\n--- FINISHED ---")
print(f"Total trips with distance > 5.0: {long_trips_count}")
print(f"Total clean rows inserted into DB: {inserted_rows}")

cur.close()
conn.close()
consumer.close()

Reading from 'green-trips-fresh' and writing to Postgres...

--- FINISHED ---
Total trips with distance > 5.0: 8506
Total clean rows inserted into DB: 49416


In [3]:
import psycopg2

conn = psycopg2.connect(
    host='localhost', port=5432, database='postgres', user='postgres', password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("""
    DROP TABLE IF EXISTS trip_counts_by_location;
    
    CREATE TABLE trip_counts_by_location (
        window_start TIMESTAMP,
        PULocationID INTEGER,
        num_trips BIGINT
    );
""")
print("Flink sink table created!")

cur.close()
conn.close()

Flink sink table created!


In [2]:
from pyflink.table import EnvironmentSettings, TableEnvironment

# 1. Set up the Flink Table Environment
env_settings = EnvironmentSettings.in_streaming_mode()
t_env = TableEnvironment.create(env_settings)

# 2. Define the Kafka Source Table (reading your JSON messages)
t_env.execute_sql("""
    CREATE TABLE green_trips_source (
        lpep_pickup_datetime STRING,
        lpep_dropoff_datetime STRING,
        PULocationID INT,
        DOLocationID INT,
        passenger_count INT,
        trip_distance DOUBLE,
        tip_amount DOUBLE,
        total_amount DOUBLE,
        -- Convert the string to a real timestamp so Flink can window it
        pickup_time AS CAST(lpep_pickup_datetime AS TIMESTAMP(3)),
        -- Tell Flink to use this time for our tumbling windows
        WATERMARK FOR pickup_time AS pickup_time - INTERVAL '5' SECOND
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'green-trips-fresh',
        'properties.bootstrap.servers' = 'localhost:9092',
        'properties.group.id' = 'flink-window-group',
        'scan.startup.mode' = 'earliest-offset',
        'format' = 'json'
    )
""")

# 3. Define the PostgreSQL Sink Table
t_env.execute_sql("""
    CREATE TABLE trip_counts_sink (
        window_start TIMESTAMP(3),
        PULocationID INT,
        num_trips BIGINT
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://localhost:5432/postgres',
        'table-name' = 'trip_counts_by_location',
        'username' = 'postgres',
        'password' = 'postgres'
    )
""")

print("Submitting the Flink Tumbling Window Job...")

# 4. Execute the Tumbling Window Aggregation!
t_env.execute_sql("""
    INSERT INTO trip_counts_sink
    SELECT 
        TUMBLE_START(pickup_time, INTERVAL '5' MINUTE) AS window_start,
        PULocationID,
        COUNT(*) AS num_trips
    FROM green_trips_source
    GROUP BY
        TUMBLE(pickup_time, INTERVAL '5' MINUTE),
        PULocationID
""").wait() # wait() keeps the cell running while Flink processes the stream

ModuleNotFoundError: No module named 'pyflink'

In [4]:
import pandas as pd

# 1. Ensure the column is a proper datetime object
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'], errors='coerce')

# 2. Drop the dirty data that was crashing Flink
clean_df = df.dropna(subset=['lpep_pickup_datetime'])

# 3. Simulate the Flink Tumbling Window (Group by 5 minutes AND Location ID)
windowed_counts = clean_df.groupby([
    pd.Grouper(key='lpep_pickup_datetime', freq='5min'), 
    'PULocationID'
]).size().reset_index(name='num_trips')

# 4. Find the top 3 busiest 5-minute windows
top_locations = windowed_counts.sort_values(by='num_trips', ascending=False).head(3)

print("Top 3 locations by 5-minute tumbling window:")
print(top_locations[['PULocationID', 'num_trips']])

KeyboardInterrupt: 

In [5]:
import pandas as pd

# 1. Clean the timestamps and sort chronologically by location
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'], errors='coerce')
clean_df = df.dropna(subset=['lpep_pickup_datetime']).sort_values(['PULocationID', 'lpep_pickup_datetime'])

# 2. Find the time gap between each consecutive pickup at the same location
clean_df['time_gap'] = clean_df.groupby('PULocationID')['lpep_pickup_datetime'].diff()

# 3. If the gap is greater than 5 minutes (or it's the very first trip), mark it as a NEW session
clean_df['is_new_session'] = (clean_df['time_gap'] > pd.Timedelta(minutes=5)) | clean_df['time_gap'].isna()

# 4. Group the data by location AND these unique sessions, then count the rows
clean_df['session_id'] = clean_df.groupby('PULocationID')['is_new_session'].cumsum()
session_counts = clean_df.groupby(['PULocationID', 'session_id']).size()

# 5. Print the absolute maximum streak!
print(f"Max trips in a single session: {session_counts.max()}")

NameError: name 'df' is not defined